# Reproduction notebook: 34A_electricity_full321_itransformer_h96_oof_pilot

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import gc
import importlib
import math
import os
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings(
    "ignore"
)

pd.set_option(
    "display.max_columns",
    260,
)

pd.set_option(
    "display.width",
    560,
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASET = "Electricity"

BACKBONES = [
    "PatchTST",
    "iTransformer",
    "TimeMixer",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

RET_SEQ_LEN = 96

TOP_K = 10
MEMORY_STRIDE = 24
OOF_ANCHOR_STRIDE = 4

FOLDS = [
    (
        0.55,
        0.70,
    ),
    (
        0.70,
        0.85,
    ),
    (
        0.85,
        1.00,
    ),
]

# Frozen predictive representation.
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DROPOUT = 0.1
REP_DIM = 64

REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

# Prospectively frozen Electricity retriever training.
# These values are declared before any Electricity test metric is evaluated.
RETRIEVER_CANDIDATE_M = 100
RETRIEVER_TAU = 0.5
RETRIEVER_LR = 1e-3
RETRIEVER_WD = 1e-4
RETRIEVER_BATCH_QUERIES = 32
RETRIEVER_MAX_EPOCHS = 30
RETRIEVER_PATIENCE = 6
RETRIEVER_GRAD_CLIP = 5.0
RETRIEVER_QUERY_STRIDE = 4
RETRIEVER_MEMORY_FRACTION = 0.60
RETRIEVER_PHASEA_TRAIN_FRACTION = 0.80
RETRIEVER_SEED = 0

# Frozen gate.
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    dtype=np.float32,
)

TARGET_RETRIEVAL_PAIRS = 672

EPS = 1e-8

RETRIEVER_USE_AMP = (
    torch.cuda.is_available()
)

CROSSFIT_SEED = 3232  # unused for training; retained for deterministic utilities

RESUME = True
FORCE = False

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_BLOCK_LEN = 24
BOOTSTRAP_SEED = 323200

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "electricity_full321_frozen_transfer_standard"
)

ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SHARED_MEMORY_EMB_DIR = (
    ROOT
    / "shared_memory_embeddings"
)

SHARED_MEMORY_EMB_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DIR = (
    ROOT
    / "artifacts"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUMMARY_PATH = (
    ROOT
    / "summary.csv"
)

BOOTSTRAP_PATH = (
    ROOT
    / "bootstrap.csv"
)

CURRENT_BACKBONE = None
DIRS = None

# Conservative direct inference blocks.
DIRECT_ANCHOR_BLOCK = {
    "PatchTST":
        2,
    "iTransformer":
        2,
    "TimeMixer":
        8,
}

SOURCE32_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "electricity_three_backbone_frozen_historical_memory"
)

SOURCE32_SUMMARY_PATH = SOURCE32_ROOT / "summary.csv"

FULL_CHANNEL_TRANSFER = True
EXPECTED_FULL_CHANNELS = 321

print(
    "Device:",
    DEVICE,
)

print(
    "Output:",
    ROOT,
)


In [ ]:
ELECTRICITY_CANDIDATES = [
    Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
    Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
    Path("/code/Time-Series-Library/dataset/electricity/electricity.csv"),
    Path("/data/dataset/electricity/electricity.csv"),
    Path("/data/dataset/electricity.csv"),
]

ELECTRICITY_PATH = next((p for p in ELECTRICITY_CANDIDATES if p.is_file()), None)
if ELECTRICITY_PATH is None:
    print("Attempted paths:")
    for p in ELECTRICITY_CANDIDATES:
        print(" -", p)
    raise FileNotFoundError("Could not find Electricity electricity.csv.")


def load_electricity_csv(path):
    df = pd.read_csv(path)

    candidate = df.copy()
    for col in list(candidate.columns):
        if str(col).lower() in {"date", "datetime", "timestamp", "time"}:
            candidate = candidate.drop(columns=[col])

    numeric = candidate.apply(pd.to_numeric, errors="coerce")
    numeric = numeric.dropna(axis=1, how="all")

    numeric = (
        numeric.replace([np.inf, -np.inf], np.nan)
        .interpolate(axis=0, limit_direction="both")
        .ffill()
        .bfill()
    )

    return numeric


raw_all_df = load_electricity_csv(ELECTRICITY_PATH)
raw = raw_all_df.to_numpy(dtype=np.float32)

n_time, n_channels = raw.shape

if n_time != 26304:
    print(
        "WARNING: standard Electricity benchmark has 26,304 rows; "
        f"loaded {n_time}."
    )

if n_channels != EXPECTED_FULL_CHANNELS:
    raise ValueError(
        "Experiment 32 is the full-channel benchmark and expects "
        f"{EXPECTED_FULL_CHANNELS} numeric channels, but loaded {n_channels}."
    )

train_end = int(0.70 * n_time)
num_test = int(0.20 * n_time)
val_end = n_time - num_test
test_end = n_time

# Standard full-channel protocol: retain all 321 benchmark channels.
train_mean = raw[:train_end].mean(axis=0).astype(np.float32)
train_std = raw[:train_end].std(axis=0, ddof=0).astype(np.float32)

if np.any(train_std <= 1e-6):
    bad = np.where(train_std <= 1e-6)[0]
    raise ValueError(
        "Standard full-channel Electricity contains train-degenerate channels "
        f"at indices {bad[:20].tolist()}. "
        "Do not silently drop them in a standard benchmark run."
    )

z_full = ((raw - train_mean[None, :]) / train_std[None, :]).astype(np.float32)

marks = np.zeros((n_time, 0), dtype=np.float32)

DATA = {
    "Electricity": {
        "name": "Electricity",
        "path": ELECTRICITY_PATH,
        "raw": raw,
        "z": z_full,
        "marks": marks,
        "n_channels": n_channels,
        "raw_channel_indices": np.arange(n_channels, dtype=np.int64),
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
    }
}

display(pd.DataFrame([
    {"Split": "Train", "Start": 0, "EndExclusive": train_end, "Length": train_end},
    {"Split": "Validation", "Start": train_end, "EndExclusive": val_end, "Length": val_end-train_end},
    {"Split": "Test", "Start": val_end, "EndExclusive": test_end, "Length": test_end-val_end},
]))

print("Electricity path:", ELECTRICITY_PATH)
print("Full benchmark shape:", raw.shape)
print("Channels:", n_channels)
print("Channel multiplier vs previous 32-channel study:", f"{n_channels / 32:.2f}x")


In [ ]:

def prefix_normalize(
    raw_array,
    prefix,
):
    prefix = int(
        prefix
    )

    mean = raw_array[
        :prefix
    ].mean(
        axis=0,
    ).astype(
        np.float32
    )

    std = raw_array[
        :prefix
    ].std(
        axis=0,
        ddof=0,
    ).astype(
        np.float32
    )

    if np.any(
        std
        <= 1e-6
    ):
        raise ValueError(
            f"Degenerate prefix channel at prefix={prefix}."
        )

    z = (
        (
            raw_array
            - mean[
                None,
                :
            ]
        )
        / std[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    return (
        z,
        {
            "Mean":
                mean,
            "Std":
                std,
            "Prefix":
                prefix,
        },
    )


def eval_anchors(
    start,
    end,
    horizon,
    stride=1,
    lookback=RET_SEQ_LEN,
):
    return np.arange(
        max(
            int(
                start
            ),
            int(
                lookback
            ),
        ),
        int(
            end
        )
        - int(
            horizon
        )
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


In [ ]:

BASE = Path(
    "/code/stock_regime_retrieval/"
    "strong_forecaster"
)

REPO_CANDIDATES = {
    "PatchTST": [
        BASE
        / "PatchTST_official",
        Path(
            "/code/PatchTST_official"
        ),
        Path(
            "/data/PatchTST_official"
        ),
    ],

    "iTransformer": [
        BASE
        / "iTransformer_official",
        Path(
            "/code/iTransformer"
        ),
        Path(
            "/data/iTransformer"
        ),
    ],

    "TimeMixer": [
        BASE
        / "TimeMixer_official",
        Path(
            "/code/TimeMixer"
        ),
        Path(
            "/data/TimeMixer"
        ),
    ],
}

REPO_URLS = {
    "PatchTST":
        "https://github.com/yuqinie98/PatchTST.git",
    "iTransformer":
        "https://github.com/thuml/iTransformer.git",
    "TimeMixer":
        "https://github.com/kwuking/TimeMixer.git",
}

EXPECTED_FILES = {
    "PatchTST":
        Path(
            "PatchTST_supervised/"
            "models/PatchTST.py"
        ),
    "iTransformer":
        Path(
            "model/"
            "iTransformer.py"
        ),
    "TimeMixer":
        Path(
            "models/"
            "TimeMixer.py"
        ),
}

REPOS = {}

for backbone in BACKBONES:
    repo = next(
        (
            p
            for p in REPO_CANDIDATES[
                backbone
            ]
            if (
                p
                / EXPECTED_FILES[
                    backbone
                ]
            ).is_file()
        ),
        None,
    )

    if repo is None:
        repo = REPO_CANDIDATES[
            backbone
        ][
            0
        ]

        repo.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            f"Cloning {backbone} -> {repo}"
        )

        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URLS[
                    backbone
                ],
                str(
                    repo
                ),
            ],
            check=True,
        )

    REPOS[
        backbone
    ] = repo

repo_rows = []

for backbone, repo in REPOS.items():
    try:
        commit = subprocess.check_output(
            [
                "git",
                "-C",
                str(
                    repo
                ),
                "rev-parse",
                "HEAD",
            ],
            text=True,
        ).strip()
    except Exception:
        commit = "unknown"

    repo_rows.append({
        "Backbone":
            backbone,
        "Repository":
            str(
                repo
            ),
        "Commit":
            commit,
    })

display(
    pd.DataFrame(
        repo_rows
    )
)

pd.DataFrame(
    repo_rows
).to_csv(
    ARTIFACT_DIR
    / "backbone_repository_commits.csv",
    index=False,
)


In [ ]:

DIRECT_RECIPES = {
    "PatchTST": {
        "seq_len":
            336,
        "label_len":
            48,
        "batch_size":
            4,
        "effective_batch_size":
            32,
        "eval_batch":
            2,
        "train_epochs":
            100,
        "patience":
            10,
        "learning_rate":
            1e-4,
        "weight_decay":
            0.0,
        "scheduler":
            "TST",
        "pct_start":
            0.2,
        "seed":
            2021,

        "e_layers":
            3,
        "n_heads":
            16,
        "d_model":
            128,
        "d_ff":
            256,
        "dropout":
            0.2,
        "fc_dropout":
            0.2,
        "head_dropout":
            0.0,
        "patch_len":
            16,
        "stride":
            8,
    },

    "iTransformer": {
        "seq_len":
            96,
        "label_len":
            48,
        "batch_size":
            2,
        "effective_batch_size":
            16,
        "eval_batch":
            2,
        "train_epochs":
            10,
        "patience":
            3,
        "learning_rate":
            5e-4,
        "weight_decay":
            0.0,
        "scheduler":
            "type1",
        "seed":
            2023,

        "e_layers":
            3,
        "n_heads":
            8,
        "d_model":
            512,
        "d_ff":
            512,
        "dropout":
            0.1,
        "factor":
            1,
    },

    "TimeMixer": {
        "seq_len":
            96,
        "label_len":
            0,
        "batch_size":
            16,
        "effective_batch_size":
            128,
        "eval_batch":
            8,
        "train_epochs":
            20,
        "patience":
            10,
        "learning_rate":
            0.01,
        "weight_decay":
            0.0,
        "scheduler":
            "OneCycle",
        "pct_start":
            0.2,
        "seed":
            2021,

        "e_layers":
            3,
        "n_heads":
            8,
        "d_model":
            16,
        "d_ff":
            32,
        "dropout":
            0.1,
        "factor":
            3,

        "down_sampling_layers":
            3,
        "down_sampling_window":
            2,
        "down_sampling_method":
            "avg",
    },
}

display(
    pd.DataFrame(
        DIRECT_RECIPES
    ).T
)


In [ ]:

ACTIVE_MODEL_CLASS = None
ACTIVE_REPO = None


def clear_forecasting_modules():
    prefixes = [
        "models",
        "model",
        "layers",
        "utils",
        "data_provider",
        "exp",
        "experiments",
    ]

    for module_name in list(
        sys.modules.keys()
    ):
        if any(
            module_name
            == p
            or module_name.startswith(
                p
                + "."
            )
            for p in prefixes
        ):
            del sys.modules[
                module_name
            ]


def activate_backbone(
    backbone,
):
    global ACTIVE_MODEL_CLASS
    global ACTIVE_REPO

    clear_forecasting_modules()

    # Remove all known forecasting repositories from sys.path.
    repo_paths = []

    for b, r in REPOS.items():
        if b == "PatchTST":
            repo_paths.append(
                str(
                    r
                    / "PatchTST_supervised"
                )
            )
        else:
            repo_paths.append(
                str(
                    r
                )
            )

    sys.path[:] = [
        p
        for p in sys.path
        if p not in repo_paths
    ]

    if backbone == "PatchTST":
        root = (
            REPOS[
                backbone
            ]
            / "PatchTST_supervised"
        )

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "models.PatchTST"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    elif backbone == "iTransformer":
        root = REPOS[
            backbone
        ]

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "model.iTransformer"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    elif backbone == "TimeMixer":
        root = REPOS[
            backbone
        ]

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "models.TimeMixer"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    else:
        raise ValueError(
            backbone
        )

    print(
        f"Activated {backbone}: "
        f"{Path(module.__file__).resolve()}"
    )

    return ACTIVE_MODEL_CLASS


In [ ]:

def set_seed(
    seed,
):
    random.seed(
        int(
            seed
        )
    )

    np.random.seed(
        int(
            seed
        )
    )

    torch.manual_seed(
        int(
            seed
        )
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(
                seed
            )
        )

    torch.backends.cudnn.benchmark = False


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def direct_config(
    backbone,
    horizon,
):
    r = DIRECT_RECIPES[
        backbone
    ]

    if backbone == "PatchTST":
        return SimpleNamespace(
            enc_in=
                n_channels,
            seq_len=
                r[
                    "seq_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            e_layers=
                r[
                    "e_layers"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            d_model=
                r[
                    "d_model"
                ],
            d_ff=
                r[
                    "d_ff"
                ],
            dropout=
                r[
                    "dropout"
                ],
            fc_dropout=
                r[
                    "fc_dropout"
                ],
            head_dropout=
                r[
                    "head_dropout"
                ],
            individual=
                0,
            patch_len=
                r[
                    "patch_len"
                ],
            stride=
                r[
                    "stride"
                ],
            padding_patch=
                "end",
            revin=
                1,
            affine=
                0,
            subtract_last=
                0,
            decomposition=
                0,
            kernel_size=
                25,
        )

    if backbone == "iTransformer":
        return SimpleNamespace(
            task_name=
                "long_term_forecast",
            seq_len=
                r[
                    "seq_len"
                ],
            label_len=
                r[
                    "label_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            enc_in=
                n_channels,
            dec_in=
                n_channels,
            c_out=
                n_channels,
            d_model=
                r[
                    "d_model"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            e_layers=
                r[
                    "e_layers"
                ],
            d_layers=
                1,
            d_ff=
                r[
                    "d_ff"
                ],
            moving_avg=
                25,
            factor=
                r[
                    "factor"
                ],
            distil=
                True,
            dropout=
                r[
                    "dropout"
                ],
            embed=
                "timeF",
            freq=
                "d",
            activation=
                "gelu",
            output_attention=
                False,
            use_norm=
                1,
            class_strategy=
                "projection",
        )

    if backbone == "TimeMixer":
        return SimpleNamespace(
            task_name=
                "long_term_forecast",
            seq_len=
                r[
                    "seq_len"
                ],
            label_len=
                r[
                    "label_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            top_k=
                5,
            num_kernels=
                6,
            enc_in=
                n_channels,
            dec_in=
                n_channels,
            c_out=
                n_channels,
            d_model=
                r[
                    "d_model"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            e_layers=
                r[
                    "e_layers"
                ],
            d_layers=
                1,
            d_ff=
                r[
                    "d_ff"
                ],
            moving_avg=
                25,
            factor=
                r[
                    "factor"
                ],
            distil=
                True,
            dropout=
                r[
                    "dropout"
                ],
            embed=
                "timeF",
            freq=
                "d",
            activation=
                "gelu",
            output_attention=
                False,
            channel_independence=
                1,
            decomp_method=
                "moving_avg",
            use_norm=
                1,
            down_sampling_layers=
                r[
                    "down_sampling_layers"
                ],
            down_sampling_window=
                r[
                    "down_sampling_window"
                ],
            down_sampling_method=
                r[
                    "down_sampling_method"
                ],
            use_future_temporal_feature=
                0,
            features=
                "M",
        )

    raise ValueError(
        backbone
    )


def build_direct_model(
    backbone,
    horizon,
):
    if ACTIVE_MODEL_CLASS is None:
        raise RuntimeError(
            "Call activate_backbone() first."
        )

    cfg = direct_config(
        backbone,
        horizon,
    )

    model = ACTIVE_MODEL_CLASS(
        cfg
    ).float().to(
        DEVICE
    )

    return (
        model,
        cfg,
    )


def direct_seq_len(
    backbone,
):
    return int(
        DIRECT_RECIPES[
            backbone
        ][
            "seq_len"
        ]
    )


def make_direct_batch(
    z,
    anchors,
    backbone,
    horizon,
):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    L = direct_seq_len(
        backbone
    )

    x_idx = (
        anchors[
            :,
            None
        ]
        - L
        + np.arange(
            L
        )[
            None,
            :
        ]
    )

    y_idx = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    x = z[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y = z[
        y_idx,
        :
    ].astype(
        np.float32
    )

    return (
        x,
        y,
    )


def forward_direct(
    backbone,
    model,
    x,
    y,
    horizon,
):
    x_t = torch.from_numpy(
        x
    ).to(
        DEVICE
    )

    y_t = torch.from_numpy(
        y
    ).to(
        DEVICE
    )

    if backbone == "PatchTST":
        pred = model(
            x_t
        )[
            :,
            -horizon:,
            :
        ]

    elif backbone == "iTransformer":
        label_len = DIRECT_RECIPES[
            backbone
        ][
            "label_len"
        ]

        dec_inp = torch.cat(
            [
                x_t[
                    :,
                    -label_len:,
                    :
                ],
                torch.zeros_like(
                    y_t
                ),
            ],
            dim=1,
        )

        pred = model(
            x_t,
            None,
            dec_inp,
            None,
        )[
            :,
            -horizon:,
            :
        ]

    elif backbone == "TimeMixer":
        pred = model(
            x_t,
            None,
            None,
            None,
        )[
            :,
            -horizon:,
            :
        ]

    else:
        raise ValueError(
            backbone
        )

    return (
        pred.float(),
        y_t.float(),
        x_t,
    )


@torch.no_grad()
def direct_residual_block(
    model,
    z,
    marks_unused,
    anchors,
    horizon,
):
    backbone = CURRENT_BACKBONE

    x, y = make_direct_batch(
        z,
        anchors,
        backbone,
        horizon,
    )

    pred, true, x_t = forward_direct(
        backbone,
        model,
        x,
        y,
        horizon,
    )

    current = x_t[
        :,
        -1:,
        :
    ].float()

    return (
        pred
        - current,
        true
        - current,
    )


In [ ]:

def backbone_dirs(
    backbone,
):
    root = (
        ROOT
        / backbone
    )

    dirs = {
        "root":
            root,
        "full_direct":
            root
            / "full_direct",
        "fold_direct":
            root
            / "fold_direct",
        "oof":
            root
            / "oof",
        "validation":
            root
            / "validation",
        "memory_emb":
            SHARED_MEMORY_EMB_DIR,
        "gate":
            root
            / "gate",
        "history":
            root
            / "history",
        "paired":
            root
            / "paired_test",
        "channel":
            root
            / "channel_test",
        "calibration":
            root
            / "calibration",
    }

    for p in dirs.values():
        if isinstance(
            p,
            Path,
        ):
            p.mkdir(
                parents=True,
                exist_ok=True,
            )

    return dirs


def full_direct_path(
    backbone,
    horizon,
):
    return (
        backbone_dirs(
            backbone
        )[
            "full_direct"
        ]
        / (
            f"Electricity_{backbone}_"
            f"H{horizon}.pt"
        )
    )


def full_direct_history_path(
    backbone,
    horizon,
):
    return (
        backbone_dirs(
            backbone
        )[
            "history"
        ]
        / (
            f"Electricity_{backbone}_"
            f"H{horizon}_full.csv"
        )
    )


def train_anchors_for_direct(
    backbone,
    boundary,
    horizon,
):
    L = direct_seq_len(
        backbone
    )

    return np.arange(
        L,
        int(
            boundary
        )
        - int(
            horizon
        )
        + 1,
        dtype=np.int64,
    )


@torch.no_grad()
def evaluate_direct_anchors(
    backbone,
    model,
    z,
    anchors,
    horizon,
    batch_size,
):
    model.eval()

    sse = 0.0
    sae = 0.0
    n = 0

    batch_mse = []

    for i in range(
        0,
        len(
            anchors
        ),
        batch_size,
    ):
        a = anchors[
            i:
            i+batch_size
        ]

        x, y = make_direct_batch(
            z,
            a,
            backbone,
            horizon,
        )

        pred, true, _ = forward_direct(
            backbone,
            model,
            x,
            y,
            horizon,
        )

        e = (
            pred
            - true
        )

        batch_mse.append(
            float(
                (
                    e
                    * e
                ).mean()
            )
        )

        sse += float(
            (
                e
                * e
            ).sum()
        )

        sae += float(
            e.abs().sum()
        )

        n += e.numel()

    return {
        "MSE":
            sse
            / n,
        "MAE":
            sae
            / n,
        "BatchAverageMSE":
            float(
                np.mean(
                    batch_mse
                )
            ),
        "Windows":
            len(
                anchors
            ),
    }


def set_epoch_lr_type1(
    optimizer,
    base_lr,
    epoch,
):
    lr = (
        base_lr
        * (
            0.5
            ** max(
                0,
                epoch
                - 1,
            )
        )
    )

    for group in optimizer.param_groups:
        group[
            "lr"
        ] = lr

    return lr


def train_or_load_full_direct(
    backbone,
    horizon,
):
    path = full_direct_path(
        backbone,
        horizon,
    )

    r = DIRECT_RECIPES[
        backbone
    ]

    model, cfg = build_direct_model(
        backbone,
        horizon,
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            f"Loaded full direct | "
            f"{backbone} H={horizon} | "
            f"best={ckpt['BestValMSE']:.6f}"
            f"@{ckpt['BestEpoch']}"
        )

        return (
            model,
            ckpt,
        )

    set_seed(
        r[
            "seed"
        ]
    )

    train_anchors = train_anchors_for_direct(
        backbone,
        train_end,
        horizon,
    )

    val_anchors = eval_anchors(
        train_end,
        val_end,
        horizon,
        stride=1,
        lookback=direct_seq_len(backbone),
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=r["learning_rate"],
        weight_decay=r["weight_decay"],
    )

    micro_batch = int(r["batch_size"])
    effective_batch = int(r.get("effective_batch_size", micro_batch))

    if effective_batch < micro_batch:
        raise ValueError(
            f"effective_batch_size ({effective_batch}) must be >= "
            f"micro batch ({micro_batch})"
        )

    scheduler = None

    if r["scheduler"] in {"TST", "OneCycle"}:
        # Scheduler steps correspond to optimizer updates, not micro-batches.
        steps_per_epoch = max(
            1,
            int(np.ceil(len(train_anchors) / effective_batch)),
        )

        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            steps_per_epoch=steps_per_epoch,
            pct_start=r["pct_start"],
            epochs=r["train_epochs"],
            max_lr=r["learning_rate"],
        )

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0
    history = []

    rng = np.random.default_rng(
        r["seed"] + horizon
    )

    print(
        f"{backbone} H={horizon} | "
        f"micro_batch={micro_batch} | "
        f"effective_batch={effective_batch} | "
        f"channels={n_channels}"
    )

    for epoch in range(
        1,
        r["train_epochs"] + 1,
    ):
        model.train()

        if r["scheduler"] == "type1":
            current_lr = set_epoch_lr_type1(
                optimizer,
                r["learning_rate"],
                epoch,
            )

        order = rng.permutation(
            train_anchors
        )

        losses = []
        t0 = time.time()

        # One optimizer update per effective batch.
        for group_left in range(
            0,
            len(order),
            effective_batch,
        ):
            group = order[
                group_left:
                group_left + effective_batch
            ]

            if len(group) == 0:
                continue

            optimizer.zero_grad(
                set_to_none=True
            )

            group_loss = 0.0
            group_n = len(group)

            for micro_left in range(
                0,
                group_n,
                micro_batch,
            ):
                a = group[
                    micro_left:
                    micro_left + micro_batch
                ]

                x, y = make_direct_batch(
                    z_full,
                    a,
                    backbone,
                    horizon,
                )

                pred, true, _ = forward_direct(
                    backbone,
                    model,
                    x,
                    y,
                    horizon,
                )

                loss = F.mse_loss(
                    pred,
                    true,
                )

                # Exact effective-batch mean, because H and C are fixed.
                weight = float(len(a)) / float(group_n)
                (loss * weight).backward()

                group_loss += float(loss.item()) * weight

                del x, y, pred, true, loss

            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            losses.append(
                group_loss
            )

        val = evaluate_direct_anchors(
            backbone,
            model,
            z_full,
            val_anchors,
            horizon,
            r["eval_batch"],
        )

        val_mse = val["MSE"]

        if (
            best_state is None
            or val_mse < best_val - 1e-12
        ):
            best_val = val_mse
            best_epoch = epoch

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            wait = 0
        else:
            wait += 1

        if scheduler is not None:
            current_lr = optimizer.param_groups[0]["lr"]

        epoch_seconds = time.time() - t0

        history.append({
            "Epoch": epoch,
            "TrainMSE": float(np.mean(losses)),
            "ValMSE": val_mse,
            "ValMAE": val["MAE"],
            "BestValMSE": best_val,
            "BestEpoch": best_epoch,
            "LR": current_lr,
            "Seconds": epoch_seconds,
            "MicroBatch": micro_batch,
            "EffectiveBatch": effective_batch,
            "Channels": n_channels,
        })

        pd.DataFrame(
            history
        ).to_csv(
            full_direct_history_path(
                backbone,
                horizon,
            ),
            index=False,
        )

        print(
            f"{backbone:12s} "
            f"H={horizon:3d} "
            f"ep={epoch:03d} | "
            f"train={history[-1]['TrainMSE']:.6f} | "
            f"val={val_mse:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"wait={wait}/{r['patience']} | "
            f"{epoch_seconds/60.0:.1f} min"
        )

        if wait >= r["patience"]:
            break

    if best_state is None:
        raise RuntimeError(
            f"No full direct checkpoint for {backbone} H={horizon}."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Backbone": backbone,
        "Dataset": "Electricity",
        "Protocol": "Full321_FrozenTransfer",
        "Horizon": int(horizon),
        "SeqLen": direct_seq_len(backbone),
        "BestEpoch": int(best_epoch),
        "BestValMSE": float(best_val),
        "Recipe": r,
        "StateDict": best_state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )


In [ ]:

def inv_softplus(
    x,
):
    return math.log(
        math.exp(
            float(
                x
            )
        )
        - 1.0
    )


class PredictivePatchEncoder(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=
                REP_D_MODEL,
            nhead=
                REP_N_HEADS,
            dim_feedforward=
                REP_D_FF,
            dropout=
                REP_DROPOUT,
            activation=
                "gelu",
            batch_first=
                True,
            norm_first=
                True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=
                REP_LAYERS,
        )

        self.norm = nn.LayerNorm(
            REP_D_MODEL
        )

        self.proj = nn.Linear(
            REP_D_MODEL,
            REP_DIM,
        )

    def forward(
        self,
        x,
    ):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(
                p
            )
            + self.pos_embed[
                :,
                :p.shape[
                    1
                ],
            ]
        )

        h = self.encoder(
            h
        ).mean(
            dim=1
        )

        h = self.proj(
            self.norm(
                h
            )
        )

        return F.normalize(
            h,
            dim=-1,
            eps=1e-8,
        )


class EmbeddingOnlyRetriever(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.encoder = (
            PredictivePatchEncoder()
        )

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(
                    1.0
                ),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(
        self,
    ):
        return F.softplus(
            self.raw_gamma
        )

    def encode(
        self,
        x,
    ):
        return self.encoder(
            x
        )


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()




STRONG_FORECASTER_ROOT = Path("/data/dataset/strong_forecaster")

EXP10_ELECTRICITY_ROOT = STRONG_FORECASTER_ROOT / "electricity_predictive_representation"
EXP12_ELECTRICITY_ROOT = STRONG_FORECASTER_ROOT / "electricity_crossfit_adaptive_gate"
EXP10_ELECTRICITY_CKPT = EXP10_ELECTRICITY_ROOT / "checkpoints"
EXP12_ELECTRICITY_FOLD_RET = EXP12_ELECTRICITY_ROOT / "fold_retriever_checkpoints"

_CKPT_RESOLVE_CACHE = {}


def _resolve_electricity_ckpt(filenames, preferred_dirs):
    key = tuple(str(x) for x in filenames)
    if key in _CKPT_RESOLVE_CACHE:
        return _CKPT_RESOLVE_CACHE[key]

    candidates = [Path(d) / fn for d in preferred_dirs for fn in filenames]
    for path in candidates:
        if path.is_file():
            _CKPT_RESOLVE_CACHE[key] = path
            return path

    matches = []
    if STRONG_FORECASTER_ROOT.is_dir():
        for fn in filenames:
            matches.extend(STRONG_FORECASTER_ROOT.rglob(fn))
    matches = sorted(set(p for p in matches if "electricity" in str(p).lower()))

    if len(matches) >= 1:
        if len(matches) > 1:
            print("WARNING: multiple Electricity checkpoint matches; using:", matches[0])
        _CKPT_RESOLVE_CACHE[key] = matches[0]
        return matches[0]

    path = candidates[0]
    _CKPT_RESOLVE_CACHE[key] = path
    return path


def full_retriever_ckpt_path(name, horizon):
    if name != "Electricity":
        raise ValueError(name)
    filenames = [
        f"Electricity_L96_H{horizon}_EmbeddingOnly_Listwise_seed0.pt",
        f"electricity_L96_H{horizon}_EmbeddingOnly_Listwise_seed0.pt",
    ]
    return _resolve_electricity_ckpt(
        filenames,
        [EXP10_ELECTRICITY_CKPT, EXP10_ELECTRICITY_ROOT],
    )


def fold_retriever_ckpt_path(name, horizon, fold):
    if name != "Electricity":
        raise ValueError(name)
    filenames = [
        f"Electricity_H{horizon}_Fold{fold}_EmbeddingOnly.pt",
        f"electricity_H{horizon}_Fold{fold}_EmbeddingOnly.pt",
    ]
    return _resolve_electricity_ckpt(
        filenames,
        [EXP12_ELECTRICITY_FOLD_RET, EXP12_ELECTRICITY_ROOT],
    )


def load_frozen_retriever(
    path,
):
    if not Path(
        path
    ).is_file():
        raise FileNotFoundError(
            path
        )

    ckpt = load_torch(
        path
    )

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    state = (
        ckpt[
            "StateDict"
        ]
        if isinstance(
            ckpt,
            dict,
        )
        and "StateDict"
        in ckpt
        else ckpt
    )

    # Backward-compatible checkpoint key handling.
    #
    # The frozen retriever checkpoints from the earlier experiments
    # were saved with:
    #   encoder.out_norm.* / encoder.out_proj.*
    # while this notebook's PredictivePatchEncoder currently exposes:
    #   encoder.norm.* / encoder.proj.*
    #
    # The modules are architecturally identical; only the attribute
    # names differ. Remap the legacy keys and still load strictly so
    # that any real architecture mismatch is not silently ignored.
    state = dict(
        state
    )

    legacy_to_current = {
        "encoder.out_norm.weight":
            "encoder.norm.weight",
        "encoder.out_norm.bias":
            "encoder.norm.bias",
        "encoder.out_proj.weight":
            "encoder.proj.weight",
        "encoder.out_proj.bias":
            "encoder.proj.bias",
    }

    for old_key, new_key in legacy_to_current.items():
        if (
            old_key in state
            and new_key not in state
        ):
            state[
                new_key
            ] = state.pop(
                old_key
            )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    for p in model.parameters():
        p.requires_grad_(
            False
        )

    return (
        model,
        ckpt,
    )


EXP10_ELECTRICITY_CKPT.mkdir(parents=True, exist_ok=True)
EXP12_ELECTRICITY_FOLD_RET.mkdir(parents=True, exist_ok=True)


In [ ]:

def retrieval_anchor_batch(
    name,
):
    C = DATA[
        name
    ][
        "n_channels"
    ]

    return max(
        1,
        TARGET_RETRIEVAL_PAIRS
        // C,
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
    horizon,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - horizon
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            horizon,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def query_pairs(
    z,
    anchors,
    channels,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    channels = np.asarray(
        channels,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        channels,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


In [ ]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e.cpu()
        )

        del (
            t,
            e,
        )

    return torch.cat(
        parts,
        dim=0,
    ).numpy().astype(
        np.float32
    )


def memory_embedding_path(
    name,
    horizon,
    tag,
):
    return (
        SHARED_MEMORY_EMB_DIR
        / (
            f"{name}_H{horizon}_"
            f"{tag}_emb.npy"
        )
    )


@torch.no_grad()
def memory_gpu_cached(
    name,
    horizon,
    tag,
    model,
    memory,
    channels,
):
    path = memory_embedding_path(
        name,
        horizon,
        tag,
    )

    expected = (
        channels,
        memory[
            "M"
        ],
        REP_DIM,
    )

    emb_np = None

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        candidate = np.load(
            path,
            mmap_mode=None,
        )

        if (
            tuple(
                candidate.shape
            )
            == expected
        ):
            emb_np = candidate.astype(
                np.float32,
                copy=False,
            )

            print(
                "Loaded memory embedding:",
                path.name,
            )

    if emb_np is None:
        emb_np = np.empty(
            expected,
            dtype=np.float32,
        )

        print(
            "Building memory embedding:",
            path.name,
            expected,
        )

        for c in range(
            channels
        ):
            emb_np[
                c
            ] = encode_np(
                model,
                memory[
                    "past"
                ][
                    c
                ],
            )

            if (
                c == 0
                or (
                    c + 1
                )
                % 50
                == 0
                or (
                    c + 1
                    == channels
                )
            ):
                print(
                    f"  channel "
                    f"{c+1}/{channels}"
                )

        np.save(
            path,
            emb_np,
        )

    return {
        "emb":
            torch.from_numpy(
                emb_np
            ).to(
                DEVICE
            ),
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


In [ ]:

@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    anchors,
    channels,
    horizon,
):
    (
        past,
        pattern,
        ctx,
        true,
    ) = query_pairs(
        z,
        anchors,
        channels,
        horizon,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        channels
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            channels
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            channels
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                channels[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }


In [ ]:

class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: "
            f"{out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


In [ ]:

@torch.no_grad()
def collect_gate_data(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    anchors,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                z,
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        # [A, H, C] -> [A, C, H]
        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                z,
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            d = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            t = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            # Strong consistency check:
            # direct true residual must match retrieval true residual.
            max_true_diff = float(
                (
                    t
                    - r[
                        "true"
                    ]
                ).abs().max()
            )

            if (
                max_true_diff
                > 2e-5
            ):
                raise RuntimeError(
                    f"True residual mismatch: "
                    f"{max_true_diff}"
                )

            feat = gate_features(
                r,
                retrieval,
                d,
            )

            abc = abc_terms(
                d,
                retrieval,
                t,
            )

            features.append(
                feat.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            abcs.append(
                abc.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            anchors_out.append(
                pair_anchor
            )

            channels_out.append(
                pair_channel
            )

            del (
                r,
                retrieval,
                d,
                t,
                feat,
                abc,
            )

        del (
            direct_big,
            true_big,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


In [ ]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
    horizon,
):
    return (
        DIRS[
            "gate"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate.pt"
        )
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()
        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    horizon,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded gate:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + horizon
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:11s} H={horizon:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and fit only on OOF for the
    # validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate_history.csv"
        ),
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )


In [ ]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_alpha_and_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    pred = (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )

    return (
        alpha,
        pred,
    )


@torch.no_grad()
def test_evaluate(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    anchors = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {
        key:
            empty_stat()
        for key in keys
    }

    anchor_mse = {
        key:
            []
        for key in keys
    }

    channel_sse_direct = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_sse_shrink = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_count = np.zeros(
        C,
        dtype=np.int64,
    )

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    oracle_alpha_sum = 0.0
    oracle_positive = 0
    n_pairs = 0

    processed = 0

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                data[
                    "z"
                ],
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        # anchor-level MSE accumulators within this direct block
        block_anchor_sums = {
            key:
                torch.zeros(
                    len(
                        a_big
                    ),
                    device=DEVICE,
                    dtype=torch.float64,
                )
            for key in keys
        }

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                data[
                    "z"
                ],
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            direct = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            true = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            scalar = (
                direct
                + scalar_alpha
                * (
                    retrieval
                    - direct
                )
            )

            features = gate_features(
                r,
                retrieval,
                direct,
            ).cpu().numpy().astype(
                np.float32
            )

            scaled = scale_features(
                features,
                gate_ckpt[
                    "FeatureMedian"
                ],
                gate_ckpt[
                    "FeatureIQR"
                ],
            )

            gate_alpha_values = gate(
                torch.from_numpy(
                    scaled
                ).to(
                    DEVICE
                )
            )

            shrink_alpha_values = (
                (
                    1.0
                    - shrink_lambda
                )
                * scalar_alpha
                + shrink_lambda
                * gate_alpha_values
            )

            raw_adaptive = (
                direct
                + gate_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            shrink_adaptive = (
                direct
                + shrink_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            (
                oracle_alpha,
                oracle,
            ) = oracle_alpha_and_prediction(
                direct,
                retrieval,
                true,
            )

            predictions = {
                "Direct":
                    direct,
                "Retrieval":
                    retrieval,
                "Scalar":
                    scalar,
                "RawAdaptive":
                    raw_adaptive,
                "ShrinkAdaptive":
                    shrink_adaptive,
                "Oracle":
                    oracle,
            }

            for key, pred in predictions.items():
                update_stat(
                    stats[
                        key
                    ],
                    pred,
                    true,
                )

                per_anchor = (
                    (
                        (
                            pred
                            - true
                        )
                        ** 2
                    )
                    .reshape(
                        A,
                        C,
                        horizon,
                    )
                    .mean(
                        dim=(
                            1,
                            2,
                        )
                    )
                    .double()
                )

                block_anchor_sums[
                    key
                ][
                    inner:
                    inner+A
                ] = per_anchor

            direct_e2 = (
                (
                    direct
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            shrink_e2 = (
                (
                    shrink_adaptive
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            channel_sse_direct += (
                direct_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_sse_shrink += (
                shrink_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_count += (
                A
                * horizon
            )

            raw_alpha_sum += float(
                gate_alpha_values.sum()
            )

            shrink_alpha_sum += float(
                shrink_alpha_values.sum()
            )

            oracle_alpha_sum += float(
                oracle_alpha.sum()
            )

            oracle_positive += int(
                (
                    oracle_alpha
                    > 0.01
                ).sum()
            )

            n_pairs += len(
                gate_alpha_values
            )

            del (
                r,
                retrieval,
                direct,
                true,
                scalar,
                features,
                scaled,
                gate_alpha_values,
                shrink_alpha_values,
                raw_adaptive,
                shrink_adaptive,
                oracle_alpha,
                oracle,
                predictions,
                direct_e2,
                shrink_e2,
            )

        for key in keys:
            anchor_mse[
                key
            ].extend(
                block_anchor_sums[
                    key
                ].cpu()
                .numpy()
                .astype(
                    np.float32
                )
                .tolist()
            )

        processed += len(
            a_big
        )

        if (
            processed
            == len(
                a_big
            )
            or processed
            % 500
            < len(
                a_big
            )
            or processed
            == len(
                anchors
            )
        ):
            print(
                f"  test anchors "
                f"{processed}/{len(anchors)}"
            )

        del (
            direct_big,
            true_big,
            block_anchor_sums,
        )

    return {
        "anchors":
            anchors,
        "metrics": {
            key:
                finish_stat(
                    value
                )
            for key, value
            in stats.items()
        },
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "channel_direct_mse":
            channel_sse_direct
            / channel_count,
        "channel_shrink_mse":
            channel_sse_shrink
            / channel_count,
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
        "oracle_mean_alpha":
            oracle_alpha_sum
            / n_pairs,
        "oracle_positive_fraction":
            oracle_positive
            / n_pairs,
    }


In [ ]:

def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=222222,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }


In [ ]:
def _ret_pattern_np(x):
    x = np.asarray(
        x,
        dtype=np.float32,
    )

    xc = x - x.mean(
        axis=-1,
        keepdims=True,
    )

    norm = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        norm > EPS,
        xc / np.maximum(
            norm,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def _ret_extract_channel(
    z,
    channel,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    pi = (
        anchors[:, None]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[None, :]
    )

    fi = (
        anchors[:, None]
        + np.arange(
            horizon
        )[None, :]
    )

    past = z[
        pi,
        channel,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channel,
    ].astype(
        np.float32
    )

    current = z[
        anchors - 1,
        channel,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[:, None]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def _ret_soft_targets(dist):
    dist = np.asarray(
        dist,
        dtype=np.float32,
    )

    mu = dist.mean(
        axis=1,
        keepdims=True,
    )

    sd = dist.std(
        axis=1,
        keepdims=True,
    )

    sd = np.maximum(
        sd,
        1e-6,
    )

    zdist = (
        dist - mu
    ) / sd

    logits = (
        -zdist
        / RETRIEVER_TAU
    )

    logits = (
        logits
        - logits.max(
            axis=1,
            keepdims=True,
        )
    )

    p = np.exp(
        logits
    ).astype(
        np.float32
    )

    p /= np.maximum(
        p.sum(
            axis=1,
            keepdims=True,
        ),
        1e-12,
    )

    return p.astype(
        np.float32
    )


# Fail early if any retriever-preparation dependency is missing.
_required_33c_retriever_helpers = [
    "_ret_extract_channel",
    "_ret_pattern_np",
    "_ret_soft_targets",
]

for _name in _required_33c_retriever_helpers:
    if _name not in globals():
        raise RuntimeError(
            f"Missing 33C retriever helper: {_name}"
        )

print(
    "PASS: 33C retriever preprocessing helpers are available."
)


In [ ]:
def c33c_balanced_pair_schedule(
    query_anchors,
    C,
    channels_per_query,
    seed,
):
    query_anchors = np.asarray(
        query_anchors,
        dtype=np.int64,
    )

    if not (
        1
        <= channels_per_query
        <= C
    ):
        raise ValueError(
            f"channels_per_query={channels_per_query}, C={C}"
        )

    rng = np.random.default_rng(seed)
    perm = rng.permutation(C).astype(np.int64)

    offsets = np.arange(
        channels_per_query,
        dtype=np.int64,
    )

    pair_anchor = np.repeat(
        query_anchors,
        channels_per_query,
    )

    pair_channel = np.empty(
        len(pair_anchor),
        dtype=np.int16,
    )

    # Step by channels_per_query. gcd(32,321)=1, so the start position
    # eventually traverses every position of the permuted channel ring.
    for j in range(len(query_anchors)):
        start = (
            j
            * channels_per_query
        ) % C

        ids = (
            start
            + offsets
        ) % C

        sl = slice(
            j * channels_per_query,
            (j + 1) * channels_per_query,
        )

        pair_channel[sl] = perm[ids].astype(
            np.int16
        )

    return pair_anchor, pair_channel


def c33c_channel_coverage_table(
    pair_anchor,
    pair_channel,
    train_mask,
    val_mask,
    C,
):
    total = np.bincount(
        pair_channel.astype(np.int64),
        minlength=C,
    )

    train = np.bincount(
        pair_channel[train_mask].astype(np.int64),
        minlength=C,
    )

    val = np.bincount(
        pair_channel[val_mask].astype(np.int64),
        minlength=C,
    )

    return pd.DataFrame({
        "Channel": np.arange(C),
        "TotalPairs": total,
        "TrainPairs": train,
        "ValPairs": val,
    })


In [ ]:
def c33c_prepare_retriever_problem(
    z,
    prefix,
    horizon,
    query_stride=RETRIEVER_QUERY_STRIDE,
):
    prefix = int(prefix)
    horizon = int(horizon)
    C = z.shape[1]

    memory_end = int(
        RETRIEVER_MEMORY_FRACTION
        * prefix
    )

    memory_end = max(
        memory_end,
        RET_SEQ_LEN
        + horizon
        + MEMORY_STRIDE,
    )

    memory_anchors = np.arange(
        RET_SEQ_LEN,
        memory_end - horizon + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(memory_anchors) < TOP_K:
        raise ValueError(
            f"Too few memory candidates: prefix={prefix}, H={horizon}"
        )

    m_use = min(
        RETRIEVER_CANDIDATE_M,
        len(memory_anchors),
    )

    q_last = prefix - horizon

    if q_last < memory_end:
        raise ValueError(
            f"No retriever supervision interval: prefix={prefix}, H={horizon}"
        )

    query_anchors = np.arange(
        memory_end,
        q_last + 1,
        query_stride,
        dtype=np.int64,
    )

    if len(query_anchors) < 4:
        raise ValueError(
            f"Too few retriever queries: H={horizon}"
        )

    pair_anchor, candidate_channel = c33c_balanced_pair_schedule(
        query_anchors,
        C,
        C33C_CHANNELS_PER_QUERY,
        C33C_CHANNEL_SCHEDULE_SEED + horizon,
    )

    n_pairs = len(pair_anchor)

    # Keep only memory past for training. Memory futures are processed
    # one channel at a time and discarded after future-distance labels
    # have been generated.
    memory_past = np.empty(
        (
            C,
            len(memory_anchors),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    query_past = np.empty(
        (
            n_pairs,
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    candidate_idx = np.empty(
        (
            n_pairs,
            m_use,
        ),
        dtype=np.int32,
    )

    future_dist = np.empty(
        (
            n_pairs,
            m_use,
        ),
        dtype=np.float32,
    )

    print(
        f"33C prep H={horizon} | "
        f"query anchors={len(query_anchors)} | "
        f"pairs={n_pairs:,} | "
        f"all-channel naive pairs={len(query_anchors)*C:,} | "
        f"sampling ratio={n_pairs/(len(query_anchors)*C):.3f}"
    )

    for c in range(C):
        mem_past_c, mem_future_c = _ret_extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        memory_past[c] = mem_past_c
        mem_pattern_c = _ret_pattern_np(
            mem_past_c
        )

        ids = np.where(
            candidate_channel.astype(np.int64)
            == c
        )[0]

        if len(ids) == 0:
            continue

        q_anchor_c = pair_anchor[ids]

        qpast, qfuture = _ret_extract_channel(
            z,
            c,
            q_anchor_c,
            horizon,
        )

        qpat = _ret_pattern_np(
            qpast
        )

        sim = (
            qpat
            @ mem_pattern_c.T
        )

        if m_use < sim.shape[1]:
            idx0 = np.argpartition(
                sim,
                -m_use,
                axis=1,
            )[:, -m_use:]

            score0 = np.take_along_axis(
                sim,
                idx0,
                axis=1,
            )

            order = np.argsort(
                -score0,
                axis=1,
            )

            idx = np.take_along_axis(
                idx0,
                order,
                axis=1,
            )
        else:
            idx = np.argsort(
                -sim,
                axis=1,
            )[:, :m_use]

        cand_future = mem_future_c[
            idx
        ]

        dist = (
            (
                cand_future
                - qfuture[:, None, :]
            ) ** 2
        ).mean(
            axis=2
        ).astype(
            np.float32
        )

        query_past[ids] = qpast
        candidate_idx[ids] = idx.astype(
            np.int32
        )
        future_dist[ids] = dist

        if (
            c == 0
            or (c + 1) % 50 == 0
            or c + 1 == C
        ):
            print(
                f"  prepared channel {c+1}/{C}"
            )

        del (
            mem_past_c,
            mem_future_c,
            mem_pattern_c,
            qpast,
            qfuture,
            qpat,
            sim,
            idx,
            cand_future,
            dist,
        )

    target_prob = _ret_soft_targets(
        future_dist
    )

    unique_a = np.unique(
        pair_anchor
    )

    cut_n = max(
        1,
        min(
            len(unique_a) - 1,
            int(
                RETRIEVER_PHASEA_TRAIN_FRACTION
                * len(unique_a)
            ),
        ),
    )

    cut_anchor = unique_a[
        cut_n
    ]

    train_mask = (
        pair_anchor
        < cut_anchor
    )

    val_mask = (
        pair_anchor
        >= cut_anchor
    )

    if (
        train_mask.sum() == 0
        or val_mask.sum() == 0
    ):
        raise RuntimeError(
            "Invalid 33C internal chronological split."
        )

    coverage = c33c_channel_coverage_table(
        pair_anchor,
        candidate_channel,
        train_mask,
        val_mask,
        C,
    )

    if (
        coverage["TrainPairs"].min() <= 0
        or coverage["ValPairs"].min() <= 0
    ):
        display(
            coverage.sort_values(
                "ValPairs"
            ).head(20)
        )
        raise RuntimeError(
            "33C channel schedule failed to expose every channel "
            "in both internal train and validation."
        )

    print(
        "Channel coverage | "
        f"train min/max={coverage['TrainPairs'].min()}/"
        f"{coverage['TrainPairs'].max()} | "
        f"val min/max={coverage['ValPairs'].min()}/"
        f"{coverage['ValPairs'].max()}"
    )

    return {
        "prefix": prefix,
        "memory_end": memory_end,
        "memory_anchors": memory_anchors,
        "memory_past": memory_past,
        "query_past": query_past,
        "candidate_idx": candidate_idx,
        "candidate_channel": candidate_channel,
        "future_dist": future_dist,
        "target_prob": target_prob,
        "pair_anchor": pair_anchor,
        "train_ids":
            np.where(train_mask)[0].astype(np.int64),
        "val_ids":
            np.where(val_mask)[0].astype(np.int64),
        "all_ids":
            np.arange(n_pairs, dtype=np.int64),
        "candidate_m": m_use,
        "coverage": coverage,
        "query_anchor_count": len(query_anchors),
    }


In [ ]:
def c33c_retriever_batch_loss(
    model,
    problem,
    ids,
):
    ids = np.asarray(
        ids,
        dtype=np.int64,
    )

    q_np = problem[
        "query_past"
    ][ids]

    ch = problem[
        "candidate_channel"
    ][ids].astype(
        np.int64
    )

    ci = problem[
        "candidate_idx"
    ][ids].astype(
        np.int64
    )

    cand_np = problem[
        "memory_past"
    ][
        ch[:, None],
        ci,
    ]

    tgt_np = problem[
        "target_prob"
    ][ids]

    q = torch.from_numpy(
        q_np
    ).to(DEVICE)

    cand = torch.from_numpy(
        cand_np.reshape(
            -1,
            RET_SEQ_LEN,
        )
    ).to(DEVICE)

    target = torch.from_numpy(
        tgt_np
    ).to(DEVICE)

    qemb = model.encode(q)

    cemb = model.encode(
        cand
    ).reshape(
        len(ids),
        problem["candidate_m"],
        REP_DIM,
    )

    score = (
        model.gamma
        * torch.einsum(
            "bd,bmd->bm",
            qemb,
            cemb,
        )
    )

    loss = -(
        target
        * F.log_softmax(
            score,
            dim=1,
        )
    ).sum(
        dim=1
    ).mean()

    return loss


def c33c_train_one_epoch(
    model,
    optimizer,
    problem,
    ids,
    rng,
):
    model.train()

    order = np.asarray(
        ids,
        dtype=np.int64,
    ).copy()

    rng.shuffle(
        order
    )

    group_losses = []

    for left in range(
        0,
        len(order),
        C33C_EFFECTIVE_BATCH_QUERIES,
    ):
        group = order[
            left:
            left + C33C_EFFECTIVE_BATCH_QUERIES
        ]

        if len(group) == 0:
            continue

        optimizer.zero_grad(
            set_to_none=True
        )

        total_group_loss = 0.0

        for mleft in range(
            0,
            len(group),
            C33C_MICRO_BATCH_QUERIES,
        ):
            micro = group[
                mleft:
                mleft
                + C33C_MICRO_BATCH_QUERIES
            ]

            loss = c33c_retriever_batch_loss(
                model,
                problem,
                micro,
            )

            weight = (
                float(len(micro))
                / float(len(group))
            )

            (
                loss
                * weight
            ).backward()

            total_group_loss += (
                float(loss.item())
                * weight
            )

            del loss

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            RETRIEVER_GRAD_CLIP,
        )

        optimizer.step()

        group_losses.append(
            total_group_loss
        )

    return float(
        np.mean(group_losses)
    )


@torch.no_grad()
def c33c_val_analog_mse(
    model,
    problem,
    ids,
):
    model.eval()

    total = 0.0
    count = 0

    for left in range(
        0,
        len(ids),
        C33C_MICRO_BATCH_QUERIES,
    ):
        bid = ids[
            left:
            left
            + C33C_MICRO_BATCH_QUERIES
        ]

        q_np = problem[
            "query_past"
        ][bid]

        ch = problem[
            "candidate_channel"
        ][bid].astype(
            np.int64
        )

        ci = problem[
            "candidate_idx"
        ][bid].astype(
            np.int64
        )

        cand_np = problem[
            "memory_past"
        ][
            ch[:, None],
            ci,
        ]

        q = torch.from_numpy(
            q_np
        ).to(DEVICE)

        cand = torch.from_numpy(
            cand_np.reshape(
                -1,
                RET_SEQ_LEN,
            )
        ).to(DEVICE)

        with ret_amp():
            qemb = model.encode(
                q
            )

            cemb = model.encode(
                cand
            ).reshape(
                len(bid),
                problem["candidate_m"],
                REP_DIM,
            )

        score = (
            model.gamma.float()
            * torch.einsum(
                "bd,bmd->bm",
                qemb.float(),
                cemb.float(),
            )
        )

        k = min(
            TOP_K,
            score.shape[1],
        )

        top = torch.topk(
            score,
            k,
            dim=1,
        ).indices.cpu().numpy()

        d = problem[
            "future_dist"
        ][bid]

        selected = np.take_along_axis(
            d,
            top,
            axis=1,
        )

        total += float(
            selected.sum()
        )

        count += selected.size

        del (
            q,
            cand,
            qemb,
            cemb,
            score,
        )

    return (
        total
        / max(
            count,
            1,
        )
    )


In [ ]:
# -------------------------------------------------------------------------
# 34A is deliberately ONE condition only.
# -------------------------------------------------------------------------
PILOT_BACKBONE = "iTransformer"
PILOT_HORIZON = 96
PILOT_FOLDS = [
    (0.55, 0.70),
    (0.70, 0.85),
    (0.85, 1.00),
]

if n_channels != 321:
    raise RuntimeError(
        f"34A requires exactly 321 Electricity channels; got {n_channels}."
    )

# Experiment 32 contains the already trained final full-321 direct baseline.
EXP32_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "electricity_full321_frozen_transfer_standard"
)

EXP32_DIRECT_PATH = (
    EXP32_ROOT
    / PILOT_BACKBONE
    / "full_direct"
    / f"Electricity_{PILOT_BACKBONE}_H{PILOT_HORIZON}.pt"
)

# Experiment 33C contains the final train-split channel-exposed retriever.
EXP33C_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "electricity_full321_channel_exposed_retriever"
)

EXP33C_FULL_RETRIEVER_PATH = (
    EXP33C_ROOT
    / "retriever"
    / (
        f"ElectricityFull321_H{PILOT_HORIZON}_"
        "ChannelExposedRetriever.pt"
    )
)

# New clean root for the OOF pilot.
EXP34A_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "electricity_full321_oof_pilot_itransformer_h96"
)
EXP34A_ROOT.mkdir(parents=True, exist_ok=True)

# All inherited path helpers below should write to the 34A root.
ROOT = EXP34A_ROOT

SHARED_MEMORY_EMB_DIR = (
    EXP34A_ROOT
    / "memory_embeddings"
)
SHARED_MEMORY_EMB_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DIR = (
    EXP34A_ROOT
    / "artifacts"
)
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CURRENT_BACKBONE = PILOT_BACKBONE
activate_backbone(
    PILOT_BACKBONE
)
DIRS = backbone_dirs(
    PILOT_BACKBONE
)

# Channel-exposed retriever recipe: same as 33C.
C33C_CHANNELS_PER_QUERY = 32
C33C_CHANNEL_SCHEDULE_SEED = 33003
C33C_EFFECTIVE_BATCH_QUERIES = 32
C33C_MICRO_BATCH_QUERIES = 8

# One-condition output.
SUMMARY_34A = EXP34A_ROOT / "summary.csv"
BOOTSTRAP_34A = EXP34A_ROOT / "bootstrap.csv"

print("34A root:", EXP34A_ROOT)
print("Pilot:", PILOT_BACKBONE, "H=", PILOT_HORIZON)
print("Channels:", n_channels)
print("Folds:", PILOT_FOLDS)

if not EXP32_DIRECT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing Experiment-32 full direct checkpoint: {EXP32_DIRECT_PATH}"
    )

direct_ckpt_audit = load_torch(
    EXP32_DIRECT_PATH
)

if (
    direct_ckpt_audit.get("Protocol")
    != "Full321_FrozenTransfer"
):
    raise RuntimeError(
        "Experiment-32 direct checkpoint protocol mismatch."
    )

print(
    "PASS: Experiment-32 full-321 iTransformer H96 checkpoint found | "
    f"best epoch={direct_ckpt_audit['BestEpoch']}"
)

if EXP33C_FULL_RETRIEVER_PATH.is_file():
    ret_ckpt_audit = load_torch(
        EXP33C_FULL_RETRIEVER_PATH
    )
    if (
        ret_ckpt_audit.get("Protocol")
        != "Full321ChannelExposedRetriever33C"
    ):
        raise RuntimeError(
            "Experiment-33C full retriever protocol mismatch."
        )
    print(
        "PASS: Experiment-33C full H96 retriever found | "
        f"best epoch={ret_ckpt_audit['BestEpoch']}"
    )
else:
    print(
        "NOTE: final 33C H96 retriever checkpoint is absent. "
        "34A will train a full-prefix fallback retriever later."
    )


In [ ]:
def load_exp32_full_direct_34a():
    model, _ = build_direct_model(
        PILOT_BACKBONE,
        PILOT_HORIZON,
    )

    ckpt = load_torch(
        EXP32_DIRECT_PATH
    )

    model.load_state_dict(
        ckpt["StateDict"],
        strict=True,
    )

    model.eval()

    print(
        "Loaded final full-321 direct | "
        f"{PILOT_BACKBONE} H={PILOT_HORIZON} | "
        f"best={ckpt['BestValMSE']:.6f}@{ckpt['BestEpoch']}"
    )

    return model, ckpt


In [ ]:
def fold_direct_path_34a(
    fold,
):
    p = (
        EXP34A_ROOT
        / "fold_direct"
    )
    p.mkdir(
        parents=True,
        exist_ok=True,
    )
    return (
        p
        / (
            f"ElectricityFull321_{PILOT_BACKBONE}_"
            f"H{PILOT_HORIZON}_F{fold}.pt"
        )
    )


def train_or_load_fold_direct_34a(
    z_fold,
    prefix,
    fold,
    fixed_epochs,
):
    path = fold_direct_path_34a(
        fold
    )

    model, _ = build_direct_model(
        PILOT_BACKBONE,
        PILOT_HORIZON,
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        if (
            ckpt.get("Protocol")
            != "Full321OOFDirect34A"
        ):
            raise RuntimeError(
                f"Invalid 34A fold direct checkpoint: {path}"
            )

        model.load_state_dict(
            ckpt["StateDict"],
            strict=True,
        )
        model.eval()

        print(
            f"Loaded fold direct F{fold} | "
            f"prefix={ckpt['Prefix']} | "
            f"epochs={ckpt['FixedEpochs']}"
        )

        return model, ckpt

    r = DIRECT_RECIPES[
        PILOT_BACKBONE
    ]

    micro_batch = int(
        r["batch_size"]
    )
    effective_batch = int(
        r["effective_batch_size"]
    )

    if effective_batch % micro_batch != 0:
        raise RuntimeError(
            "effective batch must be divisible by micro batch in 34A."
        )

    seed = (
        r["seed"]
        + 34_000
        + fold
    )
    set_seed(seed)

    anchors = train_anchors_for_direct(
        PILOT_BACKBONE,
        prefix,
        PILOT_HORIZON,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=r["learning_rate"],
        weight_decay=r["weight_decay"],
    )

    rng = np.random.default_rng(
        seed + 1
    )

    history = []

    for epoch in range(
        1,
        int(fixed_epochs) + 1,
    ):
        model.train()

        current_lr = set_epoch_lr_type1(
            optimizer,
            r["learning_rate"],
            epoch,
        )

        order = rng.permutation(
            anchors
        )

        group_losses = []
        et0 = time.time()

        for left in range(
            0,
            len(order),
            effective_batch,
        ):
            group = order[
                left:
                left + effective_batch
            ]

            if len(group) == 0:
                continue

            optimizer.zero_grad(
                set_to_none=True
            )

            group_loss = 0.0

            for mleft in range(
                0,
                len(group),
                micro_batch,
            ):
                a = group[
                    mleft:
                    mleft + micro_batch
                ]

                x, y = make_direct_batch(
                    z_fold,
                    a,
                    PILOT_BACKBONE,
                    PILOT_HORIZON,
                )

                pred, true, _ = forward_direct(
                    PILOT_BACKBONE,
                    model,
                    x,
                    y,
                    PILOT_HORIZON,
                )

                loss = F.mse_loss(
                    pred,
                    true,
                )

                weight = (
                    float(len(a))
                    / float(len(group))
                )

                (
                    loss
                    * weight
                ).backward()

                group_loss += (
                    float(loss.item())
                    * weight
                )

                del (
                    x,
                    y,
                    pred,
                    true,
                    loss,
                )

            optimizer.step()

            group_losses.append(
                group_loss
            )

        epoch_minutes = (
            time.time()
            - et0
        ) / 60.0

        history.append({
            "Epoch": epoch,
            "TrainMSE": float(
                np.mean(group_losses)
            ),
            "LR": current_lr,
            "Minutes": epoch_minutes,
        })

        print(
            f"34A FoldDirect F{fold} "
            f"ep={epoch:02d}/{fixed_epochs:02d} | "
            f"train={history[-1]['TrainMSE']:.6f} | "
            f"{epoch_minutes:.1f} min"
        )

    hist_path = (
        EXP34A_ROOT
        / "history"
    )
    hist_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame(
        history
    ).to_csv(
        hist_path
        / f"fold_direct_F{fold}.csv",
        index=False,
    )

    ckpt = {
        "Protocol":
            "Full321OOFDirect34A",
        "Backbone":
            PILOT_BACKBONE,
        "Dataset":
            "ElectricityFull321",
        "Horizon":
            PILOT_HORIZON,
        "Fold":
            int(fold),
        "Prefix":
            int(prefix),
        "FixedEpochs":
            int(fixed_epochs),
        "Seed":
            int(seed),
        "StateDict": {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    model.eval()

    return model, ckpt


In [ ]:
RET34A_DIR = (
    EXP34A_ROOT
    / "fold_retriever"
)
RET34A_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def retriever_path_34a(
    tag,
):
    return (
        RET34A_DIR
        / (
            f"ElectricityFull321_H{PILOT_HORIZON}_"
            f"{tag}_ChannelExposedRetriever34A.pt"
        )
    )


def train_or_load_retriever_34a(
    z_prefix,
    prefix,
    tag,
    seed_offset,
):
    path = retriever_path_34a(
        tag
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        if (
            ckpt.get("Protocol")
            != "Full321ChannelExposedRetriever34A"
        ):
            raise RuntimeError(
                f"Invalid 34A retriever checkpoint: {path}"
            )

        print(
            f"Loaded 34A retriever {tag} | "
            f"bestEpoch={ckpt['BestEpoch']} | "
            f"valAnalog={ckpt['BestValAnalogMSE']:.6f}"
        )

        return path, ckpt

    prep_t0 = time.time()

    problem = c33c_prepare_retriever_problem(
        z_prefix,
        prefix,
        PILOT_HORIZON,
    )

    prep_minutes = (
        time.time()
        - prep_t0
    ) / 60.0

    coverage_path = (
        RET34A_DIR
        / f"{tag}_channel_coverage.csv"
    )

    problem[
        "coverage"
    ].to_csv(
        coverage_path,
        index=False,
    )

    # -------------------------------
    # Phase A: epoch selection.
    # -------------------------------
    seed = (
        RETRIEVER_SEED
        + 34_000
        + int(seed_offset)
    )

    set_seed(seed)

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=RETRIEVER_LR,
        weight_decay=RETRIEVER_WD,
    )

    rng = np.random.default_rng(
        seed + 1
    )

    best_val = float("inf")
    best_epoch = 1
    wait = 0
    hist = []

    for epoch in range(
        1,
        RETRIEVER_MAX_EPOCHS + 1,
    ):
        et0 = time.time()

        train_loss = c33c_train_one_epoch(
            model,
            optimizer,
            problem,
            problem["train_ids"],
            rng,
        )

        val_mse = c33c_val_analog_mse(
            model,
            problem,
            problem["val_ids"],
        )

        epoch_minutes = (
            time.time()
            - et0
        ) / 60.0

        hist.append({
            "Epoch": epoch,
            "TrainLoss": train_loss,
            "ValAnalogMSE": val_mse,
            "Gamma": float(
                model.gamma.detach().cpu()
            ),
            "EpochMinutes": epoch_minutes,
        })

        print(
            f"34A Retriever {tag} "
            f"ep={epoch:02d} | "
            f"loss={train_loss:.6f} | "
            f"valAnalog={val_mse:.6f} | "
            f"{epoch_minutes:.1f} min | "
            f"best={best_val:.6f}@{best_epoch}"
        )

        if (
            val_mse
            < best_val - 1e-10
        ):
            best_val = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        if wait >= RETRIEVER_PATIENCE:
            break

    pd.DataFrame(
        hist
    ).to_csv(
        RET34A_DIR
        / f"{tag}_phaseA_history.csv",
        index=False,
    )

    del (
        model,
        optimizer,
    )
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -------------------------------
    # Phase B: refit all sampled pairs.
    # -------------------------------
    set_seed(seed)

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=RETRIEVER_LR,
        weight_decay=RETRIEVER_WD,
    )

    rng = np.random.default_rng(
        seed + 2
    )

    refit_hist = []

    for epoch in range(
        1,
        best_epoch + 1,
    ):
        et0 = time.time()

        loss = c33c_train_one_epoch(
            model,
            optimizer,
            problem,
            problem["all_ids"],
            rng,
        )

        epoch_minutes = (
            time.time()
            - et0
        ) / 60.0

        refit_hist.append({
            "Epoch": epoch,
            "TrainLoss": loss,
            "Gamma": float(
                model.gamma.detach().cpu()
            ),
            "EpochMinutes": epoch_minutes,
        })

        print(
            f"34A Refit {tag} "
            f"ep={epoch:02d}/{best_epoch:02d} | "
            f"loss={loss:.6f} | "
            f"{epoch_minutes:.1f} min"
        )

    pd.DataFrame(
        refit_hist
    ).to_csv(
        RET34A_DIR
        / f"{tag}_phaseB_history.csv",
        index=False,
    )

    total_minutes = (
        time.time()
        - prep_t0
    ) / 60.0

    ckpt = {
        "Protocol":
            "Full321ChannelExposedRetriever34A",
        "Dataset":
            "ElectricityFull321",
        "Horizon":
            PILOT_HORIZON,
        "Tag":
            str(tag),
        "Prefix":
            int(prefix),
        "BestEpoch":
            int(best_epoch),
        "BestValAnalogMSE":
            float(best_val),
        "Gamma":
            float(
                model.gamma.detach().cpu()
            ),
        "TotalTrainingMinutes":
            float(total_minutes),
        "StateDict": {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    print(
        f"SAVED 34A retriever {tag} | "
        f"bestEpoch={best_epoch} | "
        f"total={total_minutes:.1f} min"
    )

    del (
        model,
        problem,
    )
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return path, ckpt


In [ ]:
OOF34A_DIR = (
    EXP34A_ROOT
    / "oof"
)
OOF34A_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def oof_path_34a(
    fold,
):
    return (
        OOF34A_DIR
        / (
            f"ElectricityFull321_iTransformer_"
            f"H96_F{fold}_oof.npz"
        )
    )


def build_oof_fold_34a(
    fold,
    p0,
    p1,
    fixed_epochs,
):
    out_path = oof_path_34a(
        fold
    )

    if (
        RESUME
        and out_path.is_file()
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        out = {
            key: obj[key]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

        uc = np.unique(
            out["channel"]
        )

        if (
            len(uc) == 321
            and int(uc.min()) == 0
            and int(uc.max()) == 320
        ):
            print(
                f"Loaded 34A OOF F{fold}: "
                f"{len(out['abc']):,} pairs"
            )
            return out

        print(
            f"Ignoring incompatible 34A OOF cache F{fold}."
        )

    prefix = int(
        p0
        * train_end
    )

    oof_end = int(
        p1
        * train_end
    )

    z_fold, scaler = prefix_normalize(
        raw,
        prefix,
    )

    direct_model, direct_ckpt = train_or_load_fold_direct_34a(
        z_fold,
        prefix,
        fold,
        fixed_epochs,
    )

    retriever_path, retriever_ckpt = train_or_load_retriever_34a(
        z_fold,
        prefix,
        tag=f"F{fold}",
        seed_offset=fold,
    )

    retriever, _ = load_frozen_retriever(
        retriever_path
    )

    memory = build_memory(
        z_fold,
        n_channels,
        prefix,
        PILOT_HORIZON,
    )

    memory_gpu_obj = memory_gpu_cached(
        "ElectricityFull321_34A",
        PILOT_HORIZON,
        f"OOF_F{fold}_prefix{prefix}",
        retriever,
        memory,
        n_channels,
    )

    anchors = eval_anchors(
        prefix,
        oof_end,
        PILOT_HORIZON,
        stride=OOF_ANCHOR_STRIDE,
        lookback=direct_seq_len(
            PILOT_BACKBONE
        ),
    )

    print(
        f"34A OOF F{fold} | "
        f"prefix={prefix} | end={oof_end} | "
        f"anchors={len(anchors):,} | "
        f"pairs={len(anchors)*n_channels:,} | "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        DATA["Electricity"],
        PILOT_HORIZON,
        direct_model,
        retriever,
        memory_gpu_obj,
        z_fold,
        anchors,
    )

    uc = np.unique(
        out["channel"]
    )

    if len(uc) != 321:
        raise RuntimeError(
            f"OOF F{fold} missing channels: found {len(uc)}."
        )

    np.savez_compressed(
        out_path,
        **out,
    )

    print(
        f"Saved OOF F{fold}: "
        f"{len(out['abc']):,} pairs"
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        z_fold,
        scaler,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


In [ ]:
def get_full_retriever_34a():
    if EXP33C_FULL_RETRIEVER_PATH.is_file():
        ckpt = load_torch(
            EXP33C_FULL_RETRIEVER_PATH
        )

        if (
            ckpt.get("Protocol")
            != "Full321ChannelExposedRetriever33C"
        ):
            raise RuntimeError(
                "Existing 33C full retriever has an unexpected protocol."
            )

        print(
            "Reusing Experiment-33C full H96 retriever."
        )

        return (
            EXP33C_FULL_RETRIEVER_PATH,
            ckpt,
        )

    print(
        "No Experiment-33C full H96 retriever found; "
        "training 34A full-prefix fallback."
    )

    return train_or_load_retriever_34a(
        z_full,
        train_end,
        tag="FullTrain",
        seed_offset=99,
    )


VAL34A_PATH = (
    EXP34A_ROOT
    / "validation"
    / "ElectricityFull321_iTransformer_H96_validation.npz"
)
VAL34A_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def build_validation_34a(
    full_direct,
    full_retriever,
):
    if (
        RESUME
        and VAL34A_PATH.is_file()
        and not FORCE
    ):
        obj = np.load(
            VAL34A_PATH
        )

        out = {
            key: obj[key]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

        uc = np.unique(
            out["channel"]
        )

        if len(uc) == 321:
            print(
                "Loaded 34A validation cache:",
                VAL34A_PATH.name,
            )
            return out

        print(
            "Ignoring incompatible 34A validation cache."
        )

    memory = build_memory(
        z_full,
        n_channels,
        train_end,
        PILOT_HORIZON,
    )

    memory_gpu_obj = memory_gpu_cached(
        "ElectricityFull321_34A",
        PILOT_HORIZON,
        "Validation_TrainMemory",
        full_retriever,
        memory,
        n_channels,
    )

    anchors = eval_anchors(
        train_end,
        val_end,
        PILOT_HORIZON,
        stride=1,
        lookback=direct_seq_len(
            PILOT_BACKBONE
        ),
    )

    print(
        f"34A validation | anchors={len(anchors):,} | "
        f"pairs={len(anchors)*n_channels:,}"
    )

    out = collect_gate_data(
        DATA["Electricity"],
        PILOT_HORIZON,
        full_direct,
        full_retriever,
        memory_gpu_obj,
        z_full,
        anchors,
    )

    np.savez_compressed(
        VAL34A_PATH,
        **out,
    )

    del (
        memory,
        memory_gpu_obj,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


In [ ]:
t_all = time.time()

# -------------------------------------------------------------------------
# 1. Final direct model only determines the fixed fold epoch budget.
# -------------------------------------------------------------------------
full_direct, full_direct_ckpt = load_exp32_full_direct_34a()

fixed_fold_epochs = int(
    full_direct_ckpt[
        "BestEpoch"
    ]
)

if fixed_fold_epochs < 1:
    raise RuntimeError(
        f"Invalid full-direct BestEpoch: {fixed_fold_epochs}"
    )

print(
    "Fold direct fixed epoch budget:",
    fixed_fold_epochs,
)

# -------------------------------------------------------------------------
# 2. Build three OOF folds.
# -------------------------------------------------------------------------
oof_parts = []

for fold, (
    p0,
    p1,
) in enumerate(
    PILOT_FOLDS,
    start=1,
):
    fold_out = build_oof_fold_34a(
        fold,
        p0,
        p1,
        fixed_fold_epochs,
    )

    oof_parts.append(
        fold_out
    )

oof = {
    key:
        np.concatenate(
            [
                part[key]
                for part in oof_parts
            ],
            axis=0,
        )
    for key in [
        "feature",
        "abc",
        "anchor",
        "channel",
    ]
}

print(
    "Combined OOF | "
    f"pairs={len(oof['abc']):,} | "
    f"anchors={len(np.unique(oof['anchor'])):,} | "
    f"channels={len(np.unique(oof['channel']))}"
)

# -------------------------------------------------------------------------
# 3. Full-train retriever for validation/test.
# -------------------------------------------------------------------------
full_retriever_path, full_retriever_ckpt = get_full_retriever_34a()

full_retriever, _ = load_frozen_retriever(
    full_retriever_path
)

# -------------------------------------------------------------------------
# 4. Standard validation cache.
# -------------------------------------------------------------------------
val = build_validation_34a(
    full_direct,
    full_retriever,
)

# -------------------------------------------------------------------------
# 5. Cross-fitted gate.
#    OOF trains the gate; validation selects the epoch.
# -------------------------------------------------------------------------
gate, gate_ckpt = train_crossfit_gate(
    "ElectricityFull321_34A",
    PILOT_HORIZON,
    oof["feature"],
    oof["abc"],
    val["feature"],
    val["abc"],
)

# -------------------------------------------------------------------------
# 6. Validation-only scalar and shrinkage calibration.
# -------------------------------------------------------------------------
scalar_alpha, scalar_table = choose_scalar(
    val["abc"]
)

scalar_table.to_csv(
    EXP34A_ROOT
    / "validation_scalar_alpha.csv",
    index=False,
)

val_gate_alpha = gate_alpha(
    gate,
    gate_ckpt,
    val["feature"],
)

shrink_lambda, lambda_table = choose_lambda(
    val["abc"],
    val_gate_alpha,
    scalar_alpha,
)

lambda_table.to_csv(
    EXP34A_ROOT
    / "validation_shrink_lambda.csv",
    index=False,
)

val_direct_mse = mse_scalar(
    val["abc"],
    0.0,
)

val_scalar_mse = mse_scalar(
    val["abc"],
    scalar_alpha,
)

val_raw_gate_mse = mse_pair(
    val["abc"],
    val_gate_alpha,
)

val_final_alpha = (
    (1.0 - shrink_lambda)
    * scalar_alpha
    + shrink_lambda
    * val_gate_alpha
)

val_final_mse = mse_pair(
    val["abc"],
    val_final_alpha,
)

print(
    "34A validation calibration | "
    f"Direct={val_direct_mse:.6f} | "
    f"Scalar={val_scalar_mse:.6f} (a={scalar_alpha:.2f}) | "
    f"RawGate={val_raw_gate_mse:.6f} | "
    f"Final={val_final_mse:.6f} (lambda={shrink_lambda:.2f}) | "
    f"gain={100*(val_direct_mse-val_final_mse)/val_direct_mse:+.3f}%"
)

# -------------------------------------------------------------------------
# 7. Final test: memory contains train + validation only.
# -------------------------------------------------------------------------
test_memory = build_memory(
    z_full,
    n_channels,
    val_end,
    PILOT_HORIZON,
)

test_memory_gpu = memory_gpu_cached(
    "ElectricityFull321_34A",
    PILOT_HORIZON,
    "Test_TrainValMemory",
    full_retriever,
    test_memory,
    n_channels,
)

test = test_evaluate(
    DATA["Electricity"],
    PILOT_HORIZON,
    full_direct,
    full_retriever,
    test_memory_gpu,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
)

metrics = test[
    "metrics"
]

direct_mse, direct_mae = metrics[
    "Direct"
]
retrieval_mse, retrieval_mae = metrics[
    "Retrieval"
]
scalar_mse, scalar_mae = metrics[
    "Scalar"
]
raw_mse, raw_mae = metrics[
    "RawAdaptive"
]
final_mse, final_mae = metrics[
    "ShrinkAdaptive"
]
oracle_mse, oracle_mae = metrics[
    "Oracle"
]

diff = (
    test["anchor_mse"]["Direct"]
    - test["anchor_mse"]["ShrinkAdaptive"]
)

boot = moving_block_bootstrap(
    diff,
    n_boot=BOOTSTRAP_REPLICATES,
    block=BOOTSTRAP_BLOCK_LEN,
    seed=340096,
)

channel_df = pd.DataFrame({
    "Channel":
        np.arange(n_channels),
    "Direct_MSE":
        test["channel_direct_mse"],
    "OOF_Memory_MSE":
        test["channel_shrink_mse"],
})

channel_df["Gain_pct"] = (
    100.0
    * (
        channel_df["Direct_MSE"]
        - channel_df["OOF_Memory_MSE"]
    )
    / channel_df["Direct_MSE"]
)

channel_df.to_csv(
    EXP34A_ROOT
    / "channel_results.csv",
    index=False,
)

improved_channel_fraction = float(
    (
        channel_df["OOF_Memory_MSE"]
        < channel_df["Direct_MSE"]
    ).mean()
)

runtime_minutes = (
    time.time()
    - t_all
) / 60.0

summary_row = {
    "Backbone":
        PILOT_BACKBONE,
    "Dataset":
        "ElectricityFull321",
    "Horizon":
        PILOT_HORIZON,
    "Channels":
        n_channels,
    "Protocol":
        "Full321OOFCrossFitPilot34A",
    "OOFFolds":
        len(PILOT_FOLDS),
    "FoldDirectEpochs":
        fixed_fold_epochs,
    "Direct_MSE":
        direct_mse,
    "Final_MSE":
        final_mse,
    "Direct_MAE":
        direct_mae,
    "Final_MAE":
        final_mae,
    "Retrieval_MSE":
        retrieval_mse,
    "Scalar_MSE":
        scalar_mse,
    "RawAdaptive_MSE":
        raw_mse,
    "Oracle_MSE":
        oracle_mse,
    "ScalarAlpha":
        scalar_alpha,
    "ShrinkLambda":
        shrink_lambda,
    "RawMeanAlpha":
        test["raw_mean_alpha"],
    "FinalMeanAlpha":
        test["shrink_mean_alpha"],
    "Validation_Direct_MSE":
        val_direct_mse,
    "Validation_Final_MSE":
        val_final_mse,
    "Validation_Gain_pct":
        100.0
        * (
            val_direct_mse
            - val_final_mse
        )
        / val_direct_mse,
    "MSEGain_pct":
        100.0
        * (
            direct_mse
            - final_mse
        )
        / direct_mse,
    "MAEGain_pct":
        100.0
        * (
            direct_mae
            - final_mae
        )
        / direct_mae,
    "ImprovedChannelFraction":
        improved_channel_fraction,
    "OracleHeadroom_pct":
        100.0
        * (
            direct_mse
            - oracle_mse
        )
        / direct_mse,
    "GateBestEpoch":
        gate_ckpt["BestEpoch"],
    "TestAnchors":
        len(
            test["anchors"]
        ),
    "RuntimeMinutes":
        runtime_minutes,
}

pd.DataFrame(
    [summary_row]
).to_csv(
    SUMMARY_34A,
    index=False,
)

bootstrap_row = {
    "Backbone":
        PILOT_BACKBONE,
    "Dataset":
        "ElectricityFull321",
    "Horizon":
        PILOT_HORIZON,
    "Comparison":
        "Direct-OOFCrossFitMemory",
    "MeanImprovement":
        boot["MeanImprovement"],
    "CI_Low":
        boot["CI_Low"],
    "CI_High":
        boot["CI_High"],
    "SignificantPositive":
        bool(
            boot["CI_Low"] > 0
        ),
    "SignificantNegative":
        bool(
            boot["CI_High"] < 0
        ),
    "BlockLen":
        BOOTSTRAP_BLOCK_LEN,
    "Replicates":
        BOOTSTRAP_REPLICATES,
}

pd.DataFrame(
    [bootstrap_row]
).to_csv(
    BOOTSTRAP_34A,
    index=False,
)

np.savez_compressed(
    EXP34A_ROOT
    / "anchor_mse.npz",
    Anchors=test["anchors"],
    Direct=test["anchor_mse"]["Direct"],
    Retrieval=test["anchor_mse"]["Retrieval"],
    Scalar=test["anchor_mse"]["Scalar"],
    RawAdaptive=test["anchor_mse"]["RawAdaptive"],
    Final=test["anchor_mse"]["ShrinkAdaptive"],
    Oracle=test["anchor_mse"]["Oracle"],
)

print("\n" + "=" * 96)
print("EXPERIMENT 34A — FULL321 iTransformer H96 OOF PILOT")
print("=" * 96)

print(
    f"Direct MSE: {direct_mse:.6f}"
)
print(
    f"OOF Final MSE: {final_mse:.6f}"
)
print(
    f"MSE gain: {summary_row['MSEGain_pct']:+.3f}%"
)
print(
    f"95% CI Direct-Final: "
    f"[{boot['CI_Low']:.6f}, {boot['CI_High']:.6f}]"
)
print(
    "Significant positive:",
    bool(
        boot["CI_Low"] > 0
    ),
)
print(
    "Significant negative:",
    bool(
        boot["CI_High"] < 0
    ),
)
print(
    f"Scalar alpha: {scalar_alpha:.2f}"
)
print(
    f"Shrink lambda: {shrink_lambda:.2f}"
)
print(
    f"Final mean alpha: {test['shrink_mean_alpha']:.4f}"
)
print(
    f"Improved channel fraction: "
    f"{100*improved_channel_fraction:.1f}%"
)
print(
    f"Oracle headroom: "
    f"{summary_row['OracleHeadroom_pct']:.2f}%"
)
print(
    f"Runtime: {runtime_minutes:.1f} min"
)
print(
    "Saved:",
    SUMMARY_34A,
)
